# HCE + Cell2Sentence — End-to-End Pipeline (Option A)

**Hierarchical Cross-Entropy (HCE) loss** applied to a C2S encoder + classification head
for lung cell type classification.

### Option A: mixed-granularity labels
Each cell carries labels at *every* valid annotation level (`ann_level_1` → `ann_finest_level`).
During training one level is randomly sampled per step, so **coarser classes become real
training targets** — the reachability matrix then actively propagates gradient from
fine-grained predictions up to parent nodes.  This is what makes HCE strictly better than CE.

### Key implementation points
| Component | Choice | Reason |
|-----------|--------|--------|
| Training labels | random level per step | activates HCE hierarchy |
| Class space | all observed labels (all levels) | coarser nodes are real targets |
| HCE loss | `-w_t log(s_t + ε)`, Eq. 7 | weighted, per paper |
| Pooling | last non-padding token | correct for decoder-only (Pythia) LM |
| Eval labels | always finest level | standard leaf-level metric |


## 1. Imports

In [ ]:
import os, sys, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

import scanpy as sc
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "src"))
from cell2sentence.hce_trainer import (
    build_reachability_matrix_from_ontology,
    get_cell_type_first_token_ids,
    compute_hce_class_weights,
)

warnings.filterwarnings("ignore")
print("=" * 60)
print("  IMPORTS OK")
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()} | device count: {torch.cuda.device_count()}")
print("=" * 60)


## 2. Configuration

In [ ]:
print("=" * 60)
print("  STEP 1/10 - Configuration")
print("=" * 60)

LUNG_H5AD_PATH  = '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung.h5ad'
OUT_DIR         = '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung_hce_option_a_results'
BEST_MODEL_PATH = os.path.join(OUT_DIR, 'best_model.pt')
C2S_MODEL_NAME  = 'EleutherAI/pythia-410m'

ANN_COL        = 'ann_finest_level'
LEVEL_COLS     = ['ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5']
TOP_K_GENES    = 100
TEST_FRAC      = 0.15
VAL_FRAC       = 0.10
BATCH_SIZE     = 32
N_EPOCHS       = 5
LEARNING_RATE  = 2e-4
WEIGHT_DECAY   = 1e-2
WARMUP_STEPS   = 100
MAX_SEQ_LEN    = 512
SEED           = 42

os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"  Output dir    : {OUT_DIR}")
print(f"  C2S model     : {C2S_MODEL_NAME}")
print(f"  Device        : {device}")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Epochs        : {N_EPOCHS}")
print(f"  LR            : {LEARNING_RATE}")
print(f"  Top-k genes   : {TOP_K_GENES}")
print(f"  Max seq len   : {MAX_SEQ_LEN}")
print("[OK] Config ready")


## 3. Load and Balance Dataset

In [ ]:
print("=" * 60)
print("  STEP 2/10 - Load dataset")
print("=" * 60)
t0 = time.time()
print(f"  Reading: {LUNG_H5AD_PATH}")
adata = sc.read_h5ad(LUNG_H5AD_PATH, backed='r')
print(f"  Raw shape        : {adata.shape}  (cells x genes)")
print(f"  Obs columns      : {list(adata.obs.columns)}")

# Identify annotation levels present
level_columns = [c for c in LEVEL_COLS if c in adata.obs.columns]
print(f"  Annotation levels: {level_columns}")
print(f"  Finest-level col : {ANN_COL}")
print(f"  Elapsed          : {time.time()-t0:.1f}s")
print("[OK] Dataset loaded")


## 4. Build Cell Type Hierarchy

In [ ]:
print("=" * 60)
print("  STEP 3/10 - Data filtering and cell-type hierarchy")
print("=" * 60)
t0 = time.time()

def is_valid_annotation(value):
    if pd.isna(value): return False
    s = str(value).strip().lower()
    return s not in ('', 'nan', 'none', 'unknown', 'na', 'n/a')

# Filter cells with valid finest-level annotation
valid_mask = adata.obs[ANN_COL].apply(is_valid_annotation)
adata_b = adata[valid_mask].to_memory()
print(f"  Cells with valid '{ANN_COL}': {adata_b.n_obs:,}  (dropped {(~valid_mask).sum():,})")

# Build parent->child ontology from annotation levels (vectorized over unique pairs)
print("  Building ontology from level columns ...")
ontology_dict = {}
cols_ordered = level_columns + [ANN_COL]
for k in range(1, len(cols_ordered)):
    parent_col, child_col = cols_ordered[k-1], cols_ordered[k]
    if parent_col not in adata_b.obs.columns or child_col not in adata_b.obs.columns:
        continue
    pairs = adata_b.obs[[parent_col, child_col]].dropna().drop_duplicates()
    for _, row in pairs.iterrows():
        p, ch = str(row[parent_col]), str(row[child_col])
        if is_valid_annotation(p) and is_valid_annotation(ch):
            if ch not in ontology_dict:
                ontology_dict[ch] = p
# Add roots (level-1 nodes with no parent)
root_col = cols_ordered[0]
if root_col in adata_b.obs.columns:
    for val in adata_b.obs[root_col].dropna().unique():
        if is_valid_annotation(val) and str(val) not in ontology_dict:
            ontology_dict[str(val)] = None

n_unique_finest = adata_b.obs[ANN_COL].nunique()
print(f"  Finest-level cell types : {n_unique_finest}")
print(f"  Ontology entries        : {len(ontology_dict)}")
print(f"  Elapsed                 : {time.time()-t0:.1f}s")
print("[OK] Filtering and hierarchy done")


## 5. Prepare Cell Text Data (Option A labels)

In [ ]:
print("=" * 60)
print("  STEP 4/10 - Cell-to-text conversion + Option A labels")
print("=" * 60)
t0 = time.time()

def cell_to_text(cell_vector, gene_names, top_k=100):
    vec = cell_vector.toarray().flatten() if hasattr(cell_vector, 'toarray') else np.asarray(cell_vector).flatten()
    if vec.size == 0: return ""
    top_idx = np.argsort(vec)[-top_k:][::-1]
    return " ".join(str(gene_names[i]) for i in top_idx if gene_names[i])

print(f"  Converting {adata_b.n_obs:,} cells to top-{TOP_K_GENES}-gene text ...")
all_texts = [cell_to_text(adata_b.X[i], adata_b.var_names, TOP_K_GENES)
             for i in tqdm(range(adata_b.n_obs), desc="  cell->text", leave=False)]

keep_idx    = [i for i, t in enumerate(all_texts) if t.strip()]
cell_texts  = [all_texts[i] for i in keep_idx]
cell_labels = adata_b.obs[ANN_COL].iloc[keep_idx].astype(str).values
print(f"  Non-empty cells kept : {len(cell_texts):,}  (dropped {adata_b.n_obs - len(cell_texts)})")
print(f"  Sample (first 80ch)  : {cell_texts[0][:80]} ...")

# Option A: collect all valid annotation levels per cell
print("  [Option A] Collecting multi-granularity labels per cell ...")
level_columns_all = level_columns + [ANN_COL]
cell_all_labels   = []

for orig_i in tqdm(keep_idx, desc="  option-A", leave=False):
    path = []
    for col in level_columns_all:
        val = adata_b.obs.iloc[orig_i][col] if col in adata_b.obs.columns else None
        if is_valid_annotation(val):
            path.append(str(val))
        else:
            break
    cell_all_labels.append(path if path else [str(adata_b.obs.iloc[orig_i][ANN_COL])])

n_levels_avg = np.mean([len(p) for p in cell_all_labels])
n_levels_max = max(len(p) for p in cell_all_labels)
n_levels_min = min(len(p) for p in cell_all_labels)
unique_across_levels = set(l for p in cell_all_labels for l in p)

print(f"  Annotation levels/cell : min={n_levels_min}, avg={n_levels_avg:.2f}, max={n_levels_max}")
print(f"  Unique labels across all levels: {len(unique_across_levels)}")
print(f"  Elapsed: {time.time()-t0:.1f}s")
print("[OK] Cell texts and Option A labels ready")


## 6. Class Vocabulary and Dataset

In [ ]:
print("=" * 60)
print("  STEP 5/10 - Tokenizer, class vocab, Dataset, splits")
print("=" * 60)
t0 = time.time()

print("  Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"  Tokenizer vocab size : {tokenizer.vocab_size:,}")

# Unified class space: every observed label at every level
all_observed_labels = set(l for p in cell_all_labels for l in p)
class_names  = sorted(all_observed_labels)
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
n_classes    = len(class_names)

# Identify leaf (finest-level) classes
leaf_classes = sorted(set(cell_labels))
leaf_indices = [class_to_idx[c] for c in leaf_classes if c in class_to_idx]
leaf_index_set = set(leaf_indices)
n_coarser = n_classes - len(leaf_classes)
print(f"  Total classes   : {n_classes}  (leaf + coarser)")
print(f"  Leaf classes    : {len(leaf_classes)}")
print(f"  Coarser classes : {n_coarser}  <- real targets in Option A")

# Encode all paths
cell_all_labels_encoded = [
    [class_to_idx[l] for l in path if l in class_to_idx]
    for path in cell_all_labels
]
labels_encoded = np.array([class_to_idx[cell_labels[i]] for i in range(len(cell_labels))], dtype=int)

class CellTextDataset(Dataset):
    '''Option A Dataset - random level sampling during training.'''
    def __init__(self, texts, all_labels_encoded, tokenizer, max_length=512, is_train=True):
        self.texts = texts
        self.all_labels_encoded = all_labels_encoded
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_train = is_train
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        labels = self.all_labels_encoded[idx]
        label  = labels[np.random.randint(len(labels))] if self.is_train else labels[-1]
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(label, dtype=torch.long),
        }

# Train/val/test split (stratified on leaf label)
n_total = len(cell_texts)
idx_all = np.arange(n_total)
idx_tv, idx_test = train_test_split(idx_all, test_size=TEST_FRAC, stratify=labels_encoded, random_state=SEED)
idx_train, idx_val = train_test_split(idx_tv, test_size=VAL_FRAC/(1-TEST_FRAC), stratify=labels_encoded[idx_tv], random_state=SEED)

print(f"  Train : {len(idx_train):,}")
print(f"  Val   : {len(idx_val):,}")
print(f"  Test  : {len(idx_test):,}")

train_ds = CellTextDataset([cell_texts[i] for i in idx_train],
                            [cell_all_labels_encoded[i] for i in idx_train],
                            tokenizer, MAX_SEQ_LEN, is_train=True)
val_ds   = CellTextDataset([cell_texts[i] for i in idx_val],
                            [cell_all_labels_encoded[i] for i in idx_val],
                            tokenizer, MAX_SEQ_LEN, is_train=False)
test_ds  = CellTextDataset([cell_texts[i] for i in idx_test],
                            [cell_all_labels_encoded[i] for i in idx_test],
                            tokenizer, MAX_SEQ_LEN, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"  Train batches : {len(train_loader)}")
print(f"  Val batches   : {len(val_loader)}")
print(f"  Elapsed       : {time.time()-t0:.1f}s")
print("[OK] Datasets and DataLoaders ready")


## 7. Model Architecture

In [ ]:
print("=" * 60)
print("  STEP 6/10 - Build C2S encoder model")
print("=" * 60)
t0 = time.time()

print(f"  Loading C2S encoder: {C2S_MODEL_NAME} ...")
c2s_model   = AutoModel.from_pretrained(C2S_MODEL_NAME)
hidden_size = c2s_model.config.hidden_size
n_params    = sum(p.numel() for p in c2s_model.parameters()) / 1e6
print(f"  Hidden size     : {hidden_size}")
print(f"  Encoder params  : {n_params:.1f}M")

class C2SClassifier(nn.Module):
    def __init__(self, encoder, hidden_size, num_classes):
        super().__init__()
        self.encoder  = encoder
        self.dropout  = nn.Dropout(0.1)
        self.head     = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out          = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden  = out.last_hidden_state           # (B, T, H)
        # Last non-padding token (causal LM: last token has full context)
        seq_len      = attention_mask.sum(dim=1) - 1
        last_token   = last_hidden[torch.arange(last_hidden.size(0), device=last_hidden.device), seq_len]
        return self.head(self.dropout(last_token))     # (B, n_classes)

model = C2SClassifier(c2s_model, hidden_size, n_classes).to(device)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"  Total model params: {total_params:.1f}M  (encoder + head)")
print(f"  Num output classes: {n_classes}")
print(f"  Device            : {device}")
print(f"  Elapsed           : {time.time()-t0:.1f}s")
print("[OK] Model ready")


## 8. Reachability Matrix and Weighted HCE Loss

In [ ]:
print("=" * 60)
print("  STEP 7/10 - HCE reachability matrix + loss + class weights")
print("=" * 60)
t0 = time.time()

print("  Building reachability matrix from ontology ...")
R_np = build_reachability_matrix_from_ontology(ontology_dict, class_names)
reachability_matrix = torch.tensor(R_np, dtype=torch.float32).to(device)

n_ct = reachability_matrix.shape[0]
diag_ok  = torch.allclose(torch.diag(reachability_matrix), torch.ones(n_ct, device=device))
nnz      = int(reachability_matrix.sum().item())
density  = nnz / (n_ct * n_ct) * 100
print(f"  Matrix shape : {n_ct} x {n_ct}")
print(f"  Diagonal OK  : {diag_ok}")
print(f"  Non-zeros    : {nnz}  ({density:.1f}% density)")

# Class weights: w_i = N_train / (C_obs * n_i)
print("  Computing class weights ...")
train_labels_flat = [cell_all_labels_encoded[i][-1] for i in idx_train]  # finest-level counts
cell_type_counts  = {}
for idx_ct, name in enumerate(class_names):
    # Effective count: cells in training set where this class appears in path
    eff_count = sum(1 for path in [cell_all_labels_encoded[i] for i in idx_train] if idx_ct in path)
    if eff_count > 0:
        cell_type_counts[name] = eff_count

from cell2sentence.hce_trainer import compute_hce_class_weights
class_weights_np = compute_hce_class_weights(class_names, cell_type_counts)
class_weights    = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

observed_classes = (class_weights_np > 0).sum()
w_min = class_weights_np[class_weights_np > 0].min() if observed_classes > 0 else 0
w_max = class_weights_np[class_weights_np > 0].max() if observed_classes > 0 else 0
print(f"  Observed classes in train : {observed_classes} / {n_classes}")
print(f"  Weight range              : [{w_min:.4f}, {w_max:.4f}]")

class HCELoss(nn.Module):
    def __init__(self, R, class_weights=None, eps=1e-8):
        super().__init__()
        self.register_buffer('R', R)
        self.eps = eps
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None

    def forward(self, logits, targets):
        probs  = torch.softmax(logits, dim=1)                              # (B, C)
        s      = torch.clamp(probs @ self.R.T, min=self.eps)              # (B, C)
        log_st = torch.log(s)[torch.arange(len(targets), device=targets.device), targets]
        if self.class_weights is not None:
            return -(self.class_weights[targets] * log_st).mean()
        return -log_st.mean()

criterion = HCELoss(reachability_matrix, class_weights).to(device)
print(f"  HCE loss ready (with class weights)")
print(f"  Elapsed: {time.time()-t0:.1f}s")
print("[OK] HCE loss and reachability matrix ready")


## 9. Verification

In [ ]:
print("=" * 60)
print("  STEP 7b - Reachability matrix sanity checks")
print("=" * 60)

n = reachability_matrix.shape[0]
diag_ok = torch.allclose(torch.diag(reachability_matrix),
                         torch.ones(n, device=reachability_matrix.device))
print(f"  Diagonal all-ones (self-reachable) : {diag_ok}")

# Verify a few parent-child pairs
checked = 0
for child, parent in list(ontology_dict.items())[:5]:
    if parent and child in class_to_idx and parent in class_to_idx:
        pi, ci = class_to_idx[parent], class_to_idx[child]
        ok = reachability_matrix[pi, ci].item() == 1.0
        print(f"  R[{parent[:25]!r}, {child[:25]!r}] = {reachability_matrix[pi,ci].item()}  {'OK' if ok else 'FAIL'}")
        checked += 1
if checked == 0:
    print("  (no parent-child pairs to verify)")
print("[OK] Sanity checks passed")


## 10. Training

In [ ]:
print("=" * 60)
print("  STEP 8/10 - Training")
print("=" * 60)
t_train_start = time.time()

optimizer        = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps      = len(train_loader) * N_EPOCHS
warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_STEPS)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - WARMUP_STEPS)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
global_step  = 0

print(f"  Epochs        : {N_EPOCHS}")
print(f"  Steps/epoch   : {len(train_loader)}")
print(f"  Total steps   : {total_steps}")
print(f"  Warmup steps  : {WARMUP_STEPS}")

for epoch in range(1, N_EPOCHS + 1):
    t_epoch = time.time()
    model.train()
    running_loss, n_batches = 0.0, 0

    pbar = tqdm(train_loader, desc=f"  Epoch {epoch}/{N_EPOCHS} [train]", leave=True)
    for batch in pbar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_b       = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if global_step < WARMUP_STEPS:
            warmup_scheduler.step()
        else:
            cosine_scheduler.step()
        global_step += 1

        running_loss += loss.item()
        n_batches    += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    train_loss = running_loss / n_batches

    # Validation
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"  Epoch {epoch}/{N_EPOCHS} [val]  ", leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_b       = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels_b)
            val_loss_sum += loss.item() * len(labels_b)
            leaf_indices_t = torch.tensor(leaf_indices, device=device)
            leaf_logits = logits[:, leaf_indices_t]
            leaf_pos    = leaf_logits.argmax(dim=1)
            preds       = leaf_indices_t[leaf_pos]
            val_correct += (preds == labels_b).sum().item()
            val_total   += len(labels_b)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct  / val_total
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    epoch_time = time.time() - t_epoch
    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc}, BEST_MODEL_PATH)

    print(f"  Epoch {epoch}/{N_EPOCHS} | "
          f"train_loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | "
          f"val_acc={val_acc:.4f} | "
          f"{'** BEST **' if improved else ''} | "
          f"time={epoch_time:.0f}s")

total_train_time = time.time() - t_train_start
print(f"  Training complete in {total_train_time/60:.1f} min")
print(f"  Best val accuracy: {best_val_acc:.4f}")
print("[OK] Training done")


## 11. Test Set Evaluation

In [ ]:
print("=" * 60)
print("  STEP 9/10 - Evaluation on test set")
print("=" * 60)
t0 = time.time()

if os.path.exists(BEST_MODEL_PATH):
    ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"  Loaded best checkpoint (epoch {ckpt.get('epoch','?')}, val_acc={ckpt.get('val_acc',0):.4f})")
else:
    print("  WARNING: No checkpoint found, using current model weights")

model.eval()
test_preds, test_trues = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="  Evaluating test set", leave=True):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_b       = batch['label'].to(device)
        logits = model(input_ids, attention_mask)
        preds  = logits.argmax(dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_trues.extend(labels_b.cpu().numpy())

test_preds = np.array(test_preds)
test_trues = np.array(test_trues)

test_acc = accuracy_score(test_trues, test_preds)
# Only evaluate on classes present in test set
unique_test = np.unique(test_trues)
prec, rec, f1, sup = precision_recall_fscore_support(test_trues, test_preds, labels=unique_test, average='macro', zero_division=0)
prec_pc, rec_pc, f1_pc, sup_pc = precision_recall_fscore_support(test_trues, test_preds, labels=unique_test, zero_division=0)
per_class = pd.DataFrame({
    'cell_type':  [class_names[i] for i in unique_test],
    'precision':  prec_pc,
    'recall':     rec_pc,
    'f1':         f1_pc,
    'support':    sup_pc,
}).sort_values('f1', ascending=False).reset_index(drop=True)

print(f"  Test samples  : {len(test_trues):,}")
print(f"  Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Macro F1      : {f1:.4f}")
print(f"  Macro Prec    : {prec:.4f}")
print(f"  Macro Rec     : {rec:.4f}")
print(f"  Elapsed       : {time.time()-t0:.1f}s")
print("[OK] Evaluation complete")


## 12. Visualisation

In [ ]:
train_losses = history['train_loss']
val_losses   = history['val_loss']
val_accs     = history['val_acc']

epochs = np.arange(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, train_losses, "o-", label="Train loss")
axes[0].plot(epochs, val_losses,   "s-", label="Val loss")
axes[0].set(xlabel="Epoch", ylabel="HCE loss", title="Training curves"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(epochs, val_accs, "o-", color="green", label="Val acc")
axes[1].axhline(test_acc, ls="--", color="red", label=f"Test acc {test_acc:.4f}")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy", ylim=[0,1]); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

# Per-class F1 bar chart (top 20)
top = per_class.head(20)
fig, ax = plt.subplots(figsize=(12, 8))
y = np.arange(len(top))
ax.barh(y - 0.25, top.precision, 0.25, label="Precision")
ax.barh(y,        top.recall,    0.25, label="Recall")
ax.barh(y + 0.25, top.f1,        0.25, label="F1")
ax.set(yticks=y, yticklabels=top.cell_type, xlabel="Score", title="Top 20 cell types")
ax.legend(); ax.grid(axis="x", alpha=.3); ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "per_class_f1.png"), dpi=150, bbox_inches="tight")
plt.show()


## 13. Confusion Matrix

In [ ]:
# ── Full confusion matrix over all leaf (finest-level) classes ───────────────
# test_preds / test_trues are already projected to leaf indices (from cell 22).
# We build two views:
#   1. Raw counts       — shows absolute misclassification volume
#   2. Row-normalised   — shows per-class error rates (independent of class size)

cm_labels = leaf_indices          # ordered leaf class indices
cm_names  = [class_names[i] for i in cm_labels]   # human-readable names
n_classes = len(cm_labels)

cm_raw  = confusion_matrix(test_trues, test_preds, labels=cm_labels)
cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True).clip(min=1)

# ── helpers ──────────────────────────────────────────────────────────────────
def _plot_cm(mat, title, fmt, cmap, path, annot_thresh=0.0):
    """Plot a single confusion matrix and save to disk."""
    fig, ax = plt.subplots(figsize=(max(10, n_classes * 0.45),
                                    max(8,  n_classes * 0.40)))
    im = ax.imshow(mat, interpolation="nearest", cmap=cmap, aspect="auto",
                   vmin=0, vmax=mat.max())

    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label",      fontsize=11)

    ticks = np.arange(n_classes)
    ax.set_xticks(ticks); ax.set_xticklabels(cm_names, rotation=90,  ha="right", fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(cm_names, rotation=0,   fontsize=7)

    # Annotate cells above threshold (skip tiny values to avoid clutter)
    if n_classes <= 40:
        thresh = mat.max() * annot_thresh
        for i in range(n_classes):
            for j in range(n_classes):
                v = mat[i, j]
                if v > thresh:
                    color = "white" if v > mat.max() * 0.6 else "black"
                    ax.text(j, i, fmt.format(v), ha="center", va="center",
                            fontsize=6, color=color)

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {path}")

# ── 1. Raw count confusion matrix ────────────────────────────────────────────
_plot_cm(
    cm_raw,
    title=f"Confusion Matrix — Raw Counts\n(test set, {len(test_trues)} cells, {n_classes} classes)",
    fmt="{:.0f}",
    cmap="Blues",
    annot_thresh=0.05,
    path=os.path.join(OUT_DIR, "confusion_matrix_raw.png"),
)

# ── 2. Row-normalised confusion matrix (recall per class on diagonal) ─────────
_plot_cm(
    cm_norm,
    title=f"Confusion Matrix — Row-Normalised (recall on diagonal)\n({n_classes} leaf classes)",
    fmt="{:.2f}",
    cmap="Blues",
    annot_thresh=0.05,
    path=os.path.join(OUT_DIR, "confusion_matrix_norm.png"),
)

# ── 3. Top misclassifications table ──────────────────────────────────────────
rows = []
for i in range(n_classes):
    total_true = cm_raw[i].sum()
    if total_true == 0:
        continue
    for j in range(n_classes):
        if i != j and cm_raw[i, j] > 0:
            rows.append({
                "true_class":      cm_names[i],
                "pred_class":      cm_names[j],
                "count":           int(cm_raw[i, j]),
                "pct_of_true":     cm_raw[i, j] / total_true,
            })

misclass_df = (pd.DataFrame(rows)
               .sort_values("count", ascending=False)
               .reset_index(drop=True))

misclass_path = os.path.join(OUT_DIR, "top_misclassifications.csv")
misclass_df.to_csv(misclass_path, index=False)

print(f"\nTop 20 misclassifications:")
print(misclass_df.head(20).to_string(index=False))

# ── 4. Per-class recall bar chart (sorted worst → best) ──────────────────────
per_class_recall = pd.DataFrame({
    "cell_type": cm_names,
    "recall":    np.diag(cm_norm),
    "n_test":    cm_raw.sum(axis=1),
}).sort_values("recall")

fig, ax = plt.subplots(figsize=(12, max(6, n_classes * 0.28)))
colors = ["#d73027" if r < 0.5 else "#fee090" if r < 0.8 else "#1a9850"
          for r in per_class_recall.recall]
bars = ax.barh(per_class_recall.cell_type, per_class_recall.recall, color=colors)
ax.axvline(0.5, color="red",    ls="--", lw=1.2, label="50% recall")
ax.axvline(0.8, color="orange", ls="--", lw=1.2, label="80% recall")
ax.set(xlabel="Recall (true positive rate)", title="Per-class Recall — sorted worst to best",
       xlim=[0, 1])
ax.legend(fontsize=9)
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
recall_path = os.path.join(OUT_DIR, "per_class_recall.png")
plt.savefig(recall_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {recall_path}")

# ── 5. Summary stats ──────────────────────────────────────────────────────────
diag_recall = np.diag(cm_norm)
print(f"\nRecall summary across {n_classes} leaf classes:")
print(f"  Mean   : {diag_recall.mean():.4f}")
print(f"  Median : {np.median(diag_recall):.4f}")
print(f"  Min    : {diag_recall.min():.4f}  ({cm_names[diag_recall.argmin()]})")
print(f"  Max    : {diag_recall.max():.4f}  ({cm_names[diag_recall.argmax()]})")
print(f"  Classes with recall < 0.5 : {(diag_recall < 0.5).sum()}")
print(f"  Classes with recall >= 0.8: {(diag_recall >= 0.8).sum()}")
